# Multi-Asset Portfolio Performance & Risk Attribution Engine

Step 1: Install & Import Libraries
First, run this code block to install yfinance (used to pull live stock and ETF price data) and import the necessary data processing tools.

Code Explanation:

!pip install yfinance: Installs the Yahoo Finance package in your Colab environment.

import pandas as pd: Imports Pandas, which gives us DataFrames to clean, manipulate, and structure our financial time-series data.

import numpy as np: Imports NumPy, which allows us to perform vectorized mathematical operations (like calculating log returns or weighted matrix operations).


In [ ]:
# Install yfinance to download daily price data
!pip install yfinance

# Import standard libraries for data handling and calculation
import yfinance as yf
import pandas as pd
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


Step 2: Fetch Multi-Asset & Benchmark Data
Next, run this block to pull 2 years of daily adjusted closing prices for a multi-asset portfolio (Equities, Fixed Income, Gold) alongside a standard 60/40 benchmark.
Code Explanation:

Tickers:

SPY: SPDR S&P 500 ETF (represents global/US Equities).

AGG: iShares Core U.S. Aggregate Bond ETF (represents Fixed Income/Bonds).

GLD: SPDR Gold Shares (represents Commodities/Alternatives).

yf.download(...)['Close']: Pulls the daily closing prices for these three assets from January 1, 2024, to January 1, 2026.

dropna(): Ensures there are no empty trading days or missing data values, preparing a clean dataset for downstream analytics.

In [ ]:
# Define portfolio tickers and benchmark tickers
# Portfolio: Equities (SPY), Bonds (AGG), Gold (GLD)
# Benchmark: Equities (SPY), Bonds (AGG)
tickers = ['SPY', 'AGG', 'GLD']

# Fetch historical data for the past 2 years
data = yf.download(tickers, start="2024-01-01", end="2026-01-01")['Close']

# Clean data: Drop missing values
data = data.dropna()

# Display the first 5 rows of raw daily prices
data.head()

/tmp/ipykernel_1198/2972482793.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start="2024-01-01", end="2026-01-01")['Close']
[*********************100%***********************]  3 of 3 completed


Ticker,AGG,GLD,SPY
Date,,,
2024-01-02,89.123016,190.720001,458.809174
2024-01-03,89.168152,189.130005,455.062256
2024-01-04,88.807243,189.320007,453.596466
2024-01-05,88.599716,189.350006,454.217773
2024-01-08,88.933540,187.869995,460.702087


Step 3: Define Allocation Weights & Calculate Daily Returns
Now, define your actual Portfolio Allocation Weights versus your Target Benchmark Allocation Weights[cite: 1]. This step simulates an active investment decision (a "tactical tilt")[cite: 1].
Code Explanation:pct_change(): Calculates daily returns using the formula $\frac{\text{Price}_t - \text{Price}_{t-1}}{\text{Price}_{t-1}}$.Benchmark Weights: A traditional institutional benchmark consists of $60\%$ Equities (SPY) and $40\%$ Fixed Income (AGG).Portfolio Weights (Tactical Tilt): You simulate a portfolio manager who actively chooses to take on extra risk by tilting $65\%$ into Equities, $25\%$ into Bonds, and adding $10\%$ Commodities (GLD)[cite: 1].(daily_returns * w_port).sum(axis=1): Multiplies each asset's daily return by its weight and sums them together to get the total daily return of the portfolio

In [ ]:
# Calculate percentage daily returns for each asset
daily_returns = data.pct_change().dropna()

# 1. Define Benchmark Allocation Weights (60% Equities / 40% Bonds)
benchmark_weights = {
    'SPY': 0.60,
    'AGG': 0.40,
    'GLD': 0.00
}

# 2. Define Portfolio Allocation Weights (Active Tactical Tilt)
# Overweight Equities, Underweight Bonds, Add 10% Gold
portfolio_weights = {
    'SPY': 0.65,
    'AGG': 0.25,
    'GLD': 0.10
}

# Convert weights to Pandas Series
w_bench = pd.Series(benchmark_weights)
w_port = pd.Series(portfolio_weights)

# Calculate daily portfolio returns and daily benchmark returns
daily_returns['Portfolio_Return'] = (daily_returns * w_port).sum(axis=1)
daily_returns['Benchmark_Return'] = (daily_returns * w_bench).sum(axis=1)

# Display calculated returns summary
daily_returns[['Portfolio_Return', 'Benchmark_Return']].head()

Ticker,Portfolio_Return,Benchmark_Return
Date,,
2024-01-03,-0.006015,-0.004697
2024-01-04,-0.003005,-0.003552
2024-01-05,0.000322,-0.000113
2024-01-08,0.009440,0.010073
2024-01-09,-0.001005,-0.000991


Step 4: Calculate Performance Metrics & Portfolio Risk Analytics
Now, create a new code block in Colab to compute the exact institutional metrics used by the GS MAS team: Annualized Return, Annualized Volatility, Sharpe Ratio, and Tracking Error

Code ExplanationAnnualized Return: Compound growth rate scaled to a single year ($252$ trading days) using standard compounding formulas.Annualized Volatility: Measures standard deviation of daily returns scaled by $\sqrt{252}$ to quantify total portfolio risk.Sharpe Ratio: Measures risk-adjusted return relative to a baseline risk-free rate ($4.5\%$).Tracking Error: The standard deviation of the difference between portfolio and benchmark returns. It shows how closely the active portfolio follows its benchmark.  Information Ratio: Measures an active manager's ability to generate excess returns relative to the risk taken (tracking error).

In [ ]:
# 1. Trading days in a financial year
TRADING_DAYS = 252

# 2. Calculate Annualized Returns
portfolio_ann_return = (1 + daily_returns['Portfolio_Return']).prod() ** (TRADING_DAYS / len(daily_returns)) - 1
benchmark_ann_return = (1 + daily_returns['Benchmark_Return']).prod() ** (TRADING_DAYS / len(daily_returns)) - 1

# 3. Calculate Annualized Volatility (Risk)
portfolio_volatility = daily_returns['Portfolio_Return'].std() * np.sqrt(TRADING_DAYS)
benchmark_volatility = daily_returns['Benchmark_Return'].std() * np.sqrt(TRADING_DAYS)

# 4. Calculate Sharpe Ratio (Assuming 4.5% Risk-Free Rate)
risk_free_rate = 0.045
sharpe_ratio = (portfolio_ann_return - risk_free_rate) / portfolio_volatility

# 5. Calculate Tracking Error (Volatility of Excess Returns)
excess_returns = daily_returns['Portfolio_Return'] - daily_returns['Benchmark_Return']
tracking_error = excess_returns.std() * np.sqrt(TRADING_DAYS)

# 6. Calculate Information Ratio (Excess Return / Tracking Error)
information_ratio = (portfolio_ann_return - benchmark_ann_return) / tracking_error

# Create a clean summary table
metrics_summary = pd.DataFrame({
    'Metric': ['Annualized Return', 'Annualized Volatility', 'Sharpe Ratio', 'Tracking Error', 'Information Ratio'],
    'Portfolio': [f"{portfolio_ann_return:.2%}", f"{portfolio_volatility:.2%}", f"{sharpe_ratio:.2f}", f"{tracking_error:.2%}", f"{information_ratio:.2f}"],
    'Benchmark': [f"{benchmark_ann_return:.2%}", f"{benchmark_volatility:.2%}", "-", "-", "-"]
})

metrics_summary

,Metric,Portfolio,Benchmark
0,Annualized Return,19.70%,14.89%
1,Annualized Volatility,11.22%,10.26%
2,Sharpe Ratio,1.36,-
3,Tracking Error,2.00%,-
4,Information Ratio,2.41,-


Step 4: Brinson-Fachler Performance Attribution EngineThis is the core metric the GS MAS team looks for. Brinson attribution breaks down why your portfolio outperformed the benchmark:  Allocation Effect: Did you make money by overweighting/underweighting specific asset classes?  Selection Effect: Did you make money by picking better individual assets within those classes?  
Code ExplanationAllocation Effect Formula: $(w_p - w_b) \times (R_i - R_b)$.$w_p - w_b$: Your active tilt (how much you over/underweighted the asset relative to the benchmark).  $R_i - R_b$: The difference between that specific asset's return and the overall benchmark return.Why this matters for GS: It explicitly proves whether your decision to overweight Equities and add Gold created value or destroyed value relative to the standard $60/40$ benchmark.

In [ ]:
# 1. Calculate individual asset cumulative returns over the full period
asset_returns = (1 + daily_returns[['SPY', 'AGG', 'GLD']]).prod() - 1

# 2. Total benchmark return over the period
total_benchmark_return = (1 + daily_returns['Benchmark_Return']).prod() - 1

# 3. Compute Allocation Effect per asset: (w_portfolio - w_benchmark) * (R_asset - R_benchmark_total)
allocation_effect = (w_port - w_bench) * (asset_returns - total_benchmark_return)

# 4. Total Allocation & Excess Return
total_allocation_effect = allocation_effect.sum()
total_portfolio_return = (1 + daily_returns['Portfolio_Return']).prod() - 1
total_excess_return = total_portfolio_return - total_benchmark_return

# Build Attribution Summary Table
attribution_df = pd.DataFrame({
    'Asset Class': ['Equities (SPY)', 'Bonds (AGG)', 'Commodities (GLD)'],
    'Portfolio Weight': [f"{w_port['SPY']:.0%}", f"{w_port['AGG']:.0%}", f"{w_port['GLD']:.0%}"],
    'Benchmark Weight': [f"{w_bench['SPY']:.0%}", f"{w_bench['AGG']:.0%}", f"{w_bench['GLD']:.0%}"],
    'Asset Return': [f"{asset_returns['SPY']:.2%}", f"{asset_returns['AGG']:.2%}", f"{asset_returns['GLD']:.2%}"],
    'Allocation Effect': [f"{allocation_effect['SPY']:.2%}", f"{allocation_effect['AGG']:.2%}", f"{allocation_effect['GLD']:.2%}"]
})

print(f"Total Excess Return: {total_excess_return:.2%}")
print(f"Total Allocation Effect: {total_allocation_effect:.2%}\n")
attribution_df

Total Excess Return: 11.21%
Total Allocation Effect: 11.81%



,Asset Class,Portfolio Weight,Benchmark Weight,Asset Return,Allocation Effect
0,Equities (SPY),65%,60%,47.84%,0.80%
1,Bonds (AGG),25%,40%,9.11%,3.40%
2,Commodities (GLD),10%,0%,107.80%,7.60%


Step 5: Exporting SQL Database & CSVs for Visualization

Now, convert these metrics into structured, relational tables and save them as .csv files. You will use these files in SQLite/SQL queries and import them directly into Power BI to build the client portal dashboard


In [ ]:
import sqlite3

# 1. Structure Daily Returns for Database Export
daily_export = daily_returns.reset_index()
daily_export['Date'] = daily_export['Date'].dt.strftime('%Y-%m-%d')

# 2. Connect to an In-Memory SQLite Database (Simulating SQL Data Store)
conn = sqlite3.connect('portfolio_analytics.db')

# 3. Store Tables in SQL
daily_export.to_sql('fact_daily_returns', conn, if_exists='replace', index=False)
attribution_df.to_sql('fact_attribution', conn, if_exists='replace', index=False)
metrics_summary.to_sql('fact_kpi_metrics', conn, if_exists='replace', index=False)

# 4. Export CSV Files for Power BI Dashboard Import
daily_export.to_csv('fact_daily_returns.csv', index=False)
attribution_df.to_csv('fact_attribution.csv', index=False)
metrics_summary.to_csv('fact_kpi_metrics.csv', index=False)

print("SQL Database Created & CSV Files Successfully Exported!")

SQL Database Created & CSV Files Successfully Exported!


Step 6: Verify SQL Queries & Download CSV Files

Run this cell to test a sample SQL query directly in Colab and download the CSV files to your computer
Code Explanationsqlite3 Integration: Demonstrates using SQL relational logic to structure daily returns, KPI summaries, and attribution metrics into distinct database tables (fact_daily_returns, fact_attribution, fact_kpi_metrics).  files.download(...): Triggers automatic downloads of the three generated .csv files to your computer, ready to feed into Power BI.

In [ ]:
from google.colab import files

# Test SQL Execution
query_result = pd.read_sql_query("""
    SELECT Date, Portfolio_Return, Benchmark_Return
    FROM fact_daily_returns
    WHERE Portfolio_Return > Benchmark_Return
    LIMIT 5
""", conn)

print("Sample SQL Execution Test:")
print(query_result)

# Download CSVs locally for Power BI
files.download('fact_daily_returns.csv')
files.download('fact_attribution.csv')
files.download('fact_kpi_metrics.csv')

Sample SQL Execution Test:
         Date  Portfolio_Return  Benchmark_Return
0  2024-01-04         -0.003005         -0.003552
1  2024-01-05          0.000322         -0.000113
2  2024-01-10          0.002965          0.002622
3  2024-01-12          0.001885          0.001143
4  2024-01-18          0.006374          0.004969


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>